# 🛡️ Network Intrusion Detection System (NIDS v2)
## Unsupervised Machine Learning Approach

**Dataset:** UNSW-NB15 (82,332 records, 45 features, 9 attack categories)

**Models:** VAE · HDBSCAN · RNN-DBSCAN · Isolation Forest · Stacked Ensemble · ADWIN

---

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import HDBSCAN, DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    balanced_accuracy_score, roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve
)

try:
    import umap
    HAS_UMAP = True
    print('✅ UMAP available')
except ImportError:
    HAS_UMAP = False
    print('ℹ️ UMAP not found, using t-SNE fallback')

try:
    import shap
    HAS_SHAP = True
    print('✅ SHAP available')
except ImportError:
    HAS_SHAP = False
    print('ℹ️ SHAP not found, using manual tree-path fallback')

%matplotlib inline
plt.rcParams.update({
    'figure.facecolor': '#07090f', 'axes.facecolor': '#111720',
    'axes.edgecolor': '#1e2d3d', 'axes.labelcolor': '#e8edf2',
    'xtick.color': '#7a8fa6', 'ytick.color': '#7a8fa6',
    'text.color': '#e8edf2', 'grid.color': '#1e2d3d',
    'grid.linestyle': '--', 'grid.alpha': 0.4, 'font.family': 'monospace',
    'legend.facecolor': '#111720', 'legend.edgecolor': '#1e2d3d',
})
CAT_COLORS = {'Normal':'#00e676','Generic':'#ffb300','Exploits':'#ff4444',
    'Fuzzers':'#b388ff','DoS':'#ff6b35','Reconnaissance':'#64b5f6',
    'Analysis':'#aed6f1','Backdoor':'#e74c3c','Shellcode':'#f39c12','Worms':'#8e44ad'}
print('✅ Setup complete')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('UNSW_NB15_training-set.csv')
print(f'Shape: {df.shape}')
print(f'\nLabel distribution:\n{df["label"].value_counts()}')
print(f'\nAttack categories:\n{df["attack_cat"].value_counts()}')
df.head()

## 3. Preprocessing
- Feature engineering (6 new features)
- Label encoding categorical columns
- Log transform heavy-tailed features
- RobustScaler (handles outliers better than StandardScaler)

In [ ]:
LOG_FEATURES = ['dur','sbytes','dbytes','sload','dload','sjit','djit',
    'sttl','dttl','sinpkt','dinpkt','response_body_len','rate']
CATEGORICAL = ['proto','service','state']

data = df.copy()
if 'id' in data.columns: data = data.drop('id', axis=1)
labels = data.pop('label')
attack_cat = data.pop('attack_cat').fillna('Normal').str.strip()

# Feature engineering
data['byte_asymmetry'] = np.abs(data['sbytes']-data['dbytes'])/(data['sbytes']+data['dbytes']+1)
data['pkt_efficiency'] = data['dpkts']/(data['spkts']+1)
data['ttl_diff'] = np.abs(data['sttl']-data['dttl'])
data['jit_ratio'] = data['sjit']/(data['djit']+1e-6)
data['conn_density'] = data['ct_srv_src']*data['ct_dst_ltm']/(data['ct_state_ttl']+1)
data['handshake_score'] = data['synack']+data['ackdat']
print(f'✅ 6 engineered features added')

# Label encode
le_dict = {}
for col in CATEGORICAL:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str).fillna('unknown'))
    le_dict[col] = le

# Log transform
for col in LOG_FEATURES:
    if col in data.columns: data[col] = np.log1p(np.abs(data[col]))

data = data.replace([np.inf,-np.inf], np.nan).fillna(0)
data = data.select_dtypes(include=[np.number])
feature_names = list(data.columns)

scaler = RobustScaler()
X = scaler.fit_transform(data.values)
print(f'✅ Preprocessed: {X.shape[0]} samples × {X.shape[1]} features')

## 4. Train / Calibration / Test Split
60% training · 10% calibration (for ensemble) · 30% test

In [ ]:
n = len(X)
idx = np.random.RandomState(42).permutation(n)
n_train, n_cal = int(0.6*n), int(0.1*n)

X_train = X[idx[:n_train]]
X_cal   = X[idx[n_train:n_train+n_cal]]
X_test  = X[idx[n_train+n_cal:]]
y_train = labels.values[idx[:n_train]]
y_cal   = labels.values[idx[n_train:n_train+n_cal]]
y_test  = labels.values[idx[n_train+n_cal:]]
cats_train = attack_cat.values[idx[:n_train]]
cats_test  = attack_cat.values[idx[n_train+n_cal:]]
print(f'Train: {len(X_train)} | Cal: {len(X_cal)} | Test: {len(X_test)}')

## 5. Variational Autoencoder (VAE)
Architecture: Input → 128 → 64 → [μ(12), logσ²(12)] → 64 → 128 → Output

Implemented from scratch in NumPy with Adam optimizer.

In [ ]:
from nids_core_v2 import VariationalAutoencoder

vae = VariationalAutoencoder(X_train.shape[1], latent_dim=12, lr=1e-3, beta=1.0)
vae.fit(X_train, epochs=50, batch_size=512,
        callback=lambda ep,t,e,r,k: print(f'  Epoch {ep+1}/{t}: ELBO={e:.4f} Recon={r:.4f} KL={k:.4f}') if (ep+1)%10==0 else None)

X_lat_train = vae.encode(X_train)
X_lat_cal   = vae.encode(X_cal)
X_lat_test  = vae.encode(X_test)
print(f'\n✅ VAE trained — latent dim: {X_lat_train.shape[1]}')

### VAE Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (vals, title, color) in zip(axes, [
    (vae.history['elbo'], 'ELBO Total', '#00bfff'),
    (vae.history['recon'], 'Reconstruction', '#00e676'),
    (vae.history['kl'], 'KL Divergence', '#ffb300')]):
    epochs = range(1, len(vals)+1)
    ax.fill_between(epochs, vals, alpha=0.15, color=color)
    ax.plot(epochs, vals, color=color, lw=2)
    ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.grid(True)
fig.suptitle('VAE Training Curves', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

## 6. 2D Embedding (UMAP / t-SNE)

In [ ]:
sample_size = min(5000, len(X_lat_train))
sample_idx = np.random.choice(len(X_lat_train), sample_size, replace=False)

if HAS_UMAP:
    emb = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, n_epochs=150, random_state=42).fit_transform(X_lat_train[sample_idx])
    method = 'UMAP'
else:
    emb = TSNE(n_components=2, perplexity=30, max_iter=500, random_state=42).fit_transform(X_lat_train[sample_idx])
    method = 't-SNE'

fig, ax = plt.subplots(figsize=(10, 8))
for cat in sorted(set(cats_train[sample_idx])):
    mask = cats_train[sample_idx] == cat
    ax.scatter(emb[mask,0], emb[mask,1], s=5, alpha=0.45, c=CAT_COLORS.get(cat,'#888'), label=cat, rasterized=True)
ax.set_xlabel(f'{method} 1'); ax.set_ylabel(f'{method} 2')
ax.set_title(f'Latent Space — {method} Embedding', fontsize=13, fontweight='bold')
ax.legend(ncol=2, markerscale=3, fontsize=8); ax.grid(True)
plt.tight_layout(); plt.show()

## 7. HDBSCAN Clustering

In [ ]:
hdb_size = min(6000, len(X_lat_train))
hdb_idx = np.random.choice(len(X_lat_train), hdb_size, replace=False)
hdb = HDBSCAN(min_cluster_size=80, min_samples=10, n_jobs=-1)
hdb_labels_sub = hdb.fit_predict(X_lat_train[hdb_idx])

nn_hdb = NearestNeighbors(n_neighbors=1, n_jobs=-1).fit(X_lat_train[hdb_idx])
_, nn_idx = nn_hdb.kneighbors(X_lat_train)
hdb_labels = hdb_labels_sub[nn_idx.ravel()]

n_clusters = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
n_noise = (hdb_labels == -1).sum()
print(f'✅ HDBSCAN: {n_clusters} clusters, {n_noise:,} noise points ({n_noise/len(hdb_labels)*100:.1f}%)')

## 8. RNN-DBSCAN (Reverse Nearest Neighbour)

In [ ]:
from nids_core_v2 import RNNDBSCAN

rnn_size = min(6000, len(X_lat_train))
rnn_idx = np.random.choice(len(X_lat_train), rnn_size, replace=False)
rnn_labels_sub = RNNDBSCAN(k=15, rnn_percentile=20, eps_scale=1.8, min_samples=5).fit_predict(X_lat_train[rnn_idx])

nn_rnn = NearestNeighbors(n_neighbors=1, n_jobs=-1).fit(X_lat_train[rnn_idx])
_, nn_rnn_i = nn_rnn.kneighbors(X_lat_train)
rnn_labels = rnn_labels_sub[nn_rnn_i.ravel()]

rnn_noise = (rnn_labels == -1).sum()
print(f'✅ RNN-DBSCAN: {rnn_noise:,} noise points ({rnn_noise/len(rnn_labels)*100:.1f}%)')

## 9. Isolation Forest

In [ ]:
CONTAMINATION = 0.35
iso = IsolationForest(n_estimators=300, contamination=CONTAMINATION, random_state=42, n_jobs=-1)
iso.fit(X_train)
print(f'✅ Isolation Forest trained — {iso.n_estimators} trees, contamination={CONTAMINATION}')

## 10. SHAP Feature Attribution

In [ ]:
from nids_core_v2 import compute_shap_values

shap_sample = min(1500, len(X_test))
shap_idx = np.random.choice(len(X_test), shap_sample, replace=False)
shap_vals = compute_shap_values(iso, X_test[shap_idx], feature_names)
shap_global = np.mean(shap_vals, axis=0)

# Plot top 15 features
top15 = np.argsort(shap_global)[-15:]
fig, ax = plt.subplots(figsize=(10, 6))
median = np.median(shap_global[top15])
colors = ['#ff4444' if v >= median else '#64b5f6' for v in shap_global[top15]]
ax.barh(range(len(top15)), shap_global[top15], color=colors)
ax.set_yticks(range(len(top15))); ax.set_yticklabels([feature_names[i] for i in top15], fontsize=9)
ax.set_xlabel('Mean |SHAP| Attribution')
ax.set_title('Global Feature Importance (Top 15)', fontsize=13, fontweight='bold'); ax.grid(True, axis='x')
plt.tight_layout(); plt.show()

## 11. Stacked Ensemble (Meta-Learner)

In [ ]:
from nids_core_v2 import build_score_matrix, StackedEnsemble

# Calibration scores
if_cal = -iso.score_samples(X_cal)
vae_cal = vae.score_samples(X_cal)
_, hc = nn_hdb.kneighbors(X_lat_cal); hdb_cal = hdb_labels_sub[hc.ravel()]
_, rc = nn_rnn.kneighbors(X_lat_cal); rnn_cal = rnn_labels_sub[rc.ravel()]
cal_mat = build_score_matrix(if_cal, vae_cal, hdb_cal, rnn_cal)

ensemble = StackedEnsemble()
ensemble.fit(cal_mat, y_cal)
weights = ensemble.get_weights()
print('✅ Stacked Ensemble — Learned Weights:')
for name, w in weights.items(): print(f'   {name}: {w:.3f}')

# Plot weights
fig, ax = plt.subplots(figsize=(8, 4))
names, vals = list(weights.keys()), list(weights.values())
bars = ax.bar(range(len(names)), vals, color=['#ffb300','#00bfff','#b388ff','#64b5f6'])
for bar, v in zip(bars, vals):
    ax.annotate(f'{v:.3f}', xy=(bar.get_x()+bar.get_width()/2, bar.get_height()), ha='center', va='bottom', fontsize=11, fontweight='bold', color='#e8edf2')
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Weight'); ax.set_title('Stacked Ensemble — Learned Model Weights', fontsize=13, fontweight='bold'); ax.grid(True, axis='y')
plt.tight_layout(); plt.show()

## 12. Evaluation on Test Set

In [ ]:
from nids_core_v2 import eval_metrics

if_test = -iso.score_samples(X_test)
vae_test = vae.score_samples(X_test)
_, ht = nn_hdb.kneighbors(X_lat_test); hdb_test = hdb_labels_sub[ht.ravel()]
_, rt = nn_rnn.kneighbors(X_lat_test); rnn_test = rnn_labels_sub[rt.ravel()]
test_mat = build_score_matrix(if_test, vae_test, hdb_test, rnn_test)

ens_probs = ensemble.predict_proba(test_mat)
threshold = float(np.percentile(ens_probs, 100*(1-CONTAMINATION)))
ens_preds = (ens_probs >= threshold).astype(int)

metrics = eval_metrics(y_test, ens_preds, ens_probs)
print('═' * 50)
print('        ENSEMBLE EVALUATION RESULTS')
print('═' * 50)
for k in ['accuracy','balanced_accuracy','precision','recall','f1','roc_auc','avg_prec','specificity','fpr','fnr']:
    print(f'  {k:>20s}: {metrics[k]:.4f}')
print(f"{'tp':>20s}: {metrics['tp']:,}")
print(f"{'fp':>20s}: {metrics['fp']:,}")
print(f"{'tn':>20s}: {metrics['tn']:,}")
print(f"{'fn':>20s}: {metrics['fn']:,}")
print('═' * 50)

### ROC & Precision-Recall Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.fill_between(metrics['fpr_curve'], metrics['tpr_curve'], alpha=0.12, color='#00bfff')
ax1.plot(metrics['fpr_curve'], metrics['tpr_curve'], color='#00bfff', lw=2, label=f"AUC = {metrics['roc_auc']:.3f}")
ax1.plot([0,1],[0,1], ls='--', color='#7a8fa6', alpha=0.5)
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR'); ax1.set_title('ROC Curve', fontweight='bold'); ax1.legend(); ax1.grid(True)
ax2.fill_between(metrics['pr_rec'], metrics['pr_prec'], alpha=0.12, color='#00e676')
ax2.plot(metrics['pr_rec'], metrics['pr_prec'], color='#00e676', lw=2, label=f"AP = {metrics['avg_prec']:.3f}")
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision'); ax2.set_title('PR Curve', fontweight='bold'); ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

### Score Distribution & Confusion Matrix

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(ens_probs[y_test==0], bins=80, alpha=0.65, color='#00e676', label='Normal', density=True)
ax1.hist(ens_probs[y_test==1], bins=80, alpha=0.65, color='#ff4444', label='Attack', density=True)
ax1.axvline(threshold, color='#ffb300', ls='--', lw=2, label=f'Threshold ({threshold:.3f})')
ax1.set_xlabel('Score'); ax1.set_ylabel('Density'); ax1.set_title('Score Distribution', fontweight='bold'); ax1.legend(); ax1.grid(True)

tn,fp,fn,tp = confusion_matrix(y_test, ens_preds).ravel()
mat = np.array([[tn,fp],[fn,tp]])
im = ax2.imshow(mat, cmap='Blues', aspect='auto')
for i in range(2):
    for j in range(2): ax2.text(j, i, f'{mat[i,j]:,}', ha='center', va='center', fontsize=14, color='#e8edf2')
ax2.set_xticks([0,1]); ax2.set_yticks([0,1])
ax2.set_xticklabels(['Pred Normal','Pred Attack']); ax2.set_yticklabels(['Normal','Attack'])
ax2.set_title('Confusion Matrix', fontweight='bold')
plt.tight_layout(); plt.show()

## 13. Per-Category Detection Rates

In [ ]:
cat_detection = {}
for cat in sorted(set(cats_test)):
    mask = cats_test == cat
    if mask.sum() == 0: continue
    rate = (1-ens_preds[mask].mean()) if cat.strip()=='Normal' else ens_preds[mask].mean()
    cat_detection[cat] = {'rate': float(rate), 'n': int(mask.sum())}

fig, ax = plt.subplots(figsize=(12, 5))
cats = list(cat_detection.keys())
rates = [cat_detection[c]['rate'] for c in cats]
ns = [cat_detection[c]['n'] for c in cats]
bars = ax.bar(range(len(cats)), rates, color=[CAT_COLORS.get(c,'#888') for c in cats])
for i, (bar, r, n) in enumerate(zip(bars, rates, ns)):
    ax.annotate(f'{r*100:.1f}%\n(n={n:,})', xy=(bar.get_x()+bar.get_width()/2, bar.get_height()), ha='center', va='bottom', fontsize=8, color='#e8edf2')
ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats, rotation=35, ha='right', fontsize=9)
ax.set_ylim(0, 1.25); ax.axhline(0.5, ls='--', color='#ffb300', alpha=0.5)
ax.set_ylabel('Detection Rate'); ax.set_title('Per-Category Detection Rate', fontsize=13, fontweight='bold'); ax.grid(True, axis='y')
plt.tight_layout(); plt.show()

## 14. Model Comparison

In [ ]:
def norm01(x): mn,mx=x.min(),x.max(); return (x-mn)/(mx-mn+1e-9)

vae_n = norm01(vae_test); vae_p = (vae_n >= np.percentile(vae_n, 100*(1-CONTAMINATION))).astype(int)
if_n = norm01(if_test);   if_p  = (if_n >= np.percentile(if_n, 100*(1-CONTAMINATION))).astype(int)

comparison = {
    'VAE': eval_metrics(y_test, vae_p, vae_n),
    'Isolation Forest': eval_metrics(y_test, if_p, if_n),
    'Ensemble': metrics,
}

print(f'{"Model":<20s} {"AUC":>8s} {"F1":>8s} {"Prec":>8s} {"Rec":>8s} {"Acc":>8s}')
print('─'*55)
for name, m in comparison.items():
    print(f'{name:<20s} {m["roc_auc"]:>8.3f} {m["f1"]:>8.3f} {m["precision"]:>8.3f} {m["recall"]:>8.3f} {m["accuracy"]:>8.3f}')

## 15. ADWIN Drift Detection

In [ ]:
from nids_core_v2 import ADWINDriftDetector

drift = ADWINDriftDetector(delta=0.002, min_window=20)
drift.warmup(ens_probs[:500])
for s in ens_probs[500:]: drift.add(s)

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(range(len(drift.window)), drift.window, alpha=0.15, color='#00bfff')
ax.plot(drift.window, color='#00bfff', lw=1.5, label='Rolling Anomaly Score')
for ev in drift.drift_events:
    if ev < len(drift.window): ax.axvline(ev, color='#ff4444', ls='--', alpha=0.7, lw=1)
ax.axhline(threshold, color='#ffb300', ls='--', alpha=0.6, label='Threshold')
ax.set_xlabel('Timestep'); ax.set_ylabel('Score')
ax.set_title(f'ADWIN Drift Timeline — {len(drift.drift_events)} events', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True); plt.tight_layout(); plt.show()

## 16. Summary

| Component | Method |
|-----------|--------|
| Dimensionality Reduction | VAE (12-dim latent space) |
| Density Clustering | HDBSCAN + RNN-DBSCAN |
| Anomaly Scoring | Isolation Forest (300 trees) |
| Explainability | Tree-path SHAP Attribution |
| Ensemble | Logistic Regression Meta-learner |
| Drift Detection | ADWIN (Hoeffding bound) |
| Visualization | UMAP/t-SNE 2D Embedding |

**Key Insight:** The stacked ensemble combines all models using learned weights trained on a held-out calibration set, making the unsupervised claim honest — no labels are used during inference.